In [2]:
%run 10_MNESIS_polychronous-chains.ipynb

datetag = '2026-04-21'
SNNtorch version 0.9.4
Spikes in one target 1024.0,  in a SM window 42.0
for a value opt.lif_beta=0.7, the time constant is 2.8 steps
Spikes in one target 1024.0,  in a SM window 42.0
for a value opt.lif_beta=0.7, the time constant is 2.8 steps


## defining patterns as some Spiking Heidelberg Digits

In [3]:
opt = Params()
hd = HD_SNN(opt)
hd.net.to(hd.opt.device)

Sequential(
  (dropout): Dropout(p=0.37, inplace=False)
  (lin): Linear(in_features=41984, out_features=1024, bias=False)
  (lif): Leaky()
)

In [8]:
from pathlib import Path
import torch
import numpy as np
import h5py
import urllib.request
import gzip

def download_shd_dataset(data_path=Path('./shd_data')):
    """Download real SHD dataset"""
    data_path.mkdir(exist_ok=True, parents=True)
    
    # URLs for SHD dataset
    base_url = "https://datasets.codeneuro.org/datasets/shd/"
    files = [
        "shd_train.h5.gz",
        "shd_test.h5.gz"
    ]
    
    downloaded_files = []
    for file in files:
        file_path = data_path / file.replace('.gz', '')  # We'll decompress
        gz_path = data_path / file
        
        if not file_path.exists():
            print(f"Downloading {file}...")
            url = base_url + file
            try:
                urllib.request.urlretrieve(url, gz_path)
                
                # Decompress
                print(f"Decompressing {file}...")
                with gzip.open(gz_path, 'rb') as f_in:
                    with open(file_path, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
                gz_path.unlink()  # Remove compressed file
                
            except Exception as e:
                print(f"Failed to download {file}: {e}")
                continue
        
        if file_path.exists():
            downloaded_files.append(file_path)
    
    return downloaded_files

def load_shd_file(filepath):
    """Load SHD data from HDF5 file"""
    events_list = []
    labels_list = []
    
    print(f"Loading data from {filepath}")
    
    with h5py.File(filepath, 'r') as f:
        # Navigate the HDF5 structure (this may need adjustment based on actual structure)
        try:
            # Try common SHD structure
            if 'spikes' in f:
                times = f['spikes']['times'][:]
                units = f['spikes']['units'][:]
            else:
                # Alternative structure
                times = f['times'][:]
                units = f['units'][:]
            
            labels = f['labels'][:] if 'labels' in f else f['target'][:]
            
            # Process events - SHD typically has different format
            # Events are usually stored as times and neuron indices
            for i in range(len(times)):
                # Convert to event format [x, y, t, polarity]
                # This conversion depends on the exact SHD format
                events = process_shd_events(times[i], units[i])
                events_list.append(events)
                labels_list.append(labels[i])
                
        except Exception as e:
            print(f"Error reading file structure: {e}")
            # Fallback: try to read whatever keys exist
            print("Available keys:", list(f.keys()))
    
    return events_list, labels_list

def process_shd_events(times, units):
    """Convert SHD format to event format [x, y, t, polarity]"""
    # SHD stores events as (times, units/neurons)
    # Need to map neuron indices to (x,y) coordinates
    # This is approximate - you may need to adjust based on actual data
    
    if len(times) == 0:
        return np.empty((0, 4))
    
    # Assuming neurons are arranged in a grid (700 x 128)
    x_coords = units % 128  # width
    y_coords = units // 128  # height (approximate)
    y_coords = np.clip(y_coords, 0, 699)  # ensure within bounds
    
    # Polarity - SHD doesn't have polarity, so we'll set to 0
    polarity = np.zeros_like(times)
    
    # Stack into event format
    events = np.column_stack([x_coords, y_coords, times * 1000000, polarity])  # convert to microseconds
    return events

def events_to_dense(events, time_steps=100, height=700, width=128):
    """Convert events to dense tensor"""
    if len(events) == 0:
        return torch.zeros(time_steps, 1, height, width)  # 1 channel since no polarity in SHD
    
    # Sort events by time
    events_sorted = events[events[:, 2].argsort()]
    
    # Create time bins
    max_time = events_sorted[:, 2].max() if len(events_sorted) > 0 else 1
    time_bins = np.linspace(0, max_time, time_steps + 1)
    
    # Initialize dense tensor (1 channel for SHD)
    dense_tensor = torch.zeros(time_steps, 1, height, width)
    
    # Bin events into time steps
    for event in events_sorted:
        x, y, t, p = event.astype(int)
        time_bin = np.digitize(t, time_bins) - 1
        time_bin = np.clip(time_bin, 0, time_steps - 1)
        
        if 0 <= x < width and 0 <= y < height:
            dense_tensor[time_bin, 0, y, x] += 1  # Only 1 channel
    
    return dense_tensor

def get_real_shd_as_dense(data_path=Path('./shd_data')):
    """Get real SHD dataset as dense tensors"""
    # Download dataset
    files = download_shd_dataset(data_path)
    
    all_dense_samples = []
    all_labels = []
    
    # Process each file
    for filepath in files:
        print(f"Processing {filepath.name}...")
        try:
            events_list, labels_list = load_shd_file(filepath)
            
            # Convert to dense tensors
            for events, label in zip(events_list, labels_list):
                dense_tensor = events_to_dense(events, time_steps=100)
                all_dense_samples.append(dense_tensor)
                all_labels.append(label)
                
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            continue
    
    if all_dense_samples:
        return torch.stack(all_dense_samples), torch.tensor(all_labels)
    else:
        raise ValueError("No data loaded successfully")

# Usage
try:
    print("Loading real SHD dataset...")
    data_path = Path('./shd_data')
    dense_data, labels = get_real_shd_as_dense(data_path)
    
    print(f"Dense data shape: {dense_data.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Number of samples: {len(labels)}")
    print(f"Unique labels: {torch.unique(labels)}")
    
except Exception as e:
    print(f"Error loading SHD dataset: {e}")
    print("Falling back to synthetic data for demonstration...")
    
    # Synthetic fallback
    def create_synthetic_shd_fallback(num_samples=10, time_steps=100, height=700, width=128):
        dense_samples = []
        labels = []
        for i in range(num_samples):
            # Create simple pattern
            sample = torch.zeros(time_steps, 1, height, width)
            # Add some random events
            for _ in range(100):
                t = np.random.randint(0, time_steps)
                y = np.random.randint(0, height)
                x = np.random.randint(0, width)
                sample[t, 0, y, x] = 1
            dense_samples.append(sample)
            labels.append(i % 10)  # 10 digit classes
        return torch.stack(dense_samples), torch.tensor(labels)
    
    dense_data, labels = create_synthetic_shd_fallback()
    print(f"Fallback - Dense data shape: {dense_data.shape}")

Loading real SHD dataset...
Failed to download shd_train.h5.gz: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)>
Failed to download shd_test.h5.gz: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)>
Error loading SHD dataset: No data loaded successfully
Falling back to synthetic data for demonstration...
Fallback - Dense data shape: torch.Size([10, 100, 1, 700, 128])


In [ ]:

# Make the target periodic
for i_SM, angle in enumerate(np.linspace(0, 2*np.pi, opt.N_SM, endpoint=False)):
    TW = get_TW_spike(angle=angle)
    hd.target[i_SM, :, :] = TW.reshape((opt.N_neuron, opt.N_time))
hd.target.shape

## learning Spiking Heidelberg Digits patterns

In [ ]:
model_filename = data_cache / f"{hd.opt.datetag}_shd.pth"
lock_filename = data_cache / model_filename.with_suffix('.lock')
# if False:
if RECOMPUTE:
    model_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
try:
    model_state_dict = torch.load(model_filename, map_location=torch.device(hd.opt.device))
    hd.net.load_state_dict(model_state_dict)
    hd.net.eval()
    lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
    print(f"Model weights loaded from {model_filename}") # Add a log message
except FileNotFoundError:
    if not lock_filename.exists():
        print(f"Model file not found: {model_filename}, intitializing the new model.")
        lock_filename.touch(exist_ok=True)
        ##################
        hd.update_weight()
        hd.learn_model()
        ##################
        torch.save(hd.net.state_dict(), model_filename)
        lock_filename.unlink(missing_ok=True)
    else:
        print(f"Model file is locked: {lock_filename}, passing.")

In [ ]:
with torch.no_grad():
    target_full = torch.nn.functional.pad(hd.target, (opt.N_pretime, opt.N_pretime))
    input_spikes = hd.get_input_spikes(p_A=hd.opt.p_A, N_pretime=hd.opt.N_pretime, N_trigger_time=hd.opt.num_delay)
    _, _, spikes = hd.forward_pass(input_spikes)
    spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(-hd.opt.N_pretime)]
    target_evoked = hd.target[:, :, hd.opt.num_delay:]

    precision, recall, f1_score = get_scores(spikes_evoked, target_evoked)
    print(f'precision = {precision:.3f}\t recall = {recall:.3f}\t f1_score = {f1_score:.3f} ')

In [ ]:
fig,ax = plt.subplots(figsize=(13, 8))
splt.raster(spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="blue", marker='+', alpha=.5)
splt.raster(target_full[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
if figpath is not None: printfig(fig, 'target', fig_width=opt.fig_width, fig_height=opt.fig_height)
spikes.shape, spikes[i_SM, :, :].sum().item(), hd.target[i_SM, :, :].sum().item()

In [ ]:
# hd.net.lin.bias.cpu().detach().numpy()
fig,ax = plt.subplots(figsize=(13, 8))
ax.hist(hd.net.lin.weight.cpu().detach().numpy().ravel(), bins=100, density=True) # pyright: ignore[reportCallIssue, reportAttributeAccessIssue]
ax.set_yscale('log')